# Vids-2 Google Drive Direct Indexer (Colab Edition)

This notebook runs crawling and batch indexing using **Google Colab's free CPU, RAM, and network bandwidth**.
Index splits are written **directly into your Google Drive**.
**Zero ingestion traffic touches Render**, keeping your Render service dedicated strictly to fast user searches.

### Step 1: Install Quickwit & Rclone

In [ ]:
!curl -s https://rclone.org/install.sh | sudo bash
!curl -sL https://github.com/quickwit-oss/quickwit/releases/download/v0.8.2/quickwit-v0.8.2-x86_64-unknown-linux-gnu.tar.gz | tar -xz -C /tmp
!sudo mv /tmp/quickwit-v0.8.2/quickwit /usr/local/bin/quickwit
!quickwit --version && rclone version

### Step 2: Configure Google Drive Token for Rclone
Paste your rclone Google Drive JSON token below.

In [ ]:
import os

# Set your rclone credentials
os.environ['RCLONE_CONFIG_GDRIVE_TYPE'] = 'drive'
os.environ['RCLONE_CONFIG_GDRIVE_SCOPE'] = 'drive'
# Paste your rclone token JSON string here:
GDRIVE_TOKEN = '{"access_token":"...","refresh_token":"...","token_type":"Bearer"}'
os.environ['RCLONE_CONFIG_GDRIVE_TOKEN'] = GDRIVE_TOKEN

!mkdir -p /root/.config/rclone
with open('/root/.config/rclone/rclone.conf', 'w') as f:
    f.write(f'''[gdrive]
type = drive
scope = drive
token = {GDRIVE_TOKEN}
''')

# Test connection to Google Drive
!rclone about gdrive:

### Step 3: Start the Background Rclone S3 Bridge & Setup Quickwit Config

In [ ]:
# Start rclone S3 gateway in background
!mkdir -p /tmp/qwdata
!nohup rclone serve s3 --addr 127.0.0.1:9099 --auth-key fake,fake --dir-cache-time 5s gdrive:quickwit-data > /tmp/rclone_s3.log 2>&1 &

# Create Quickwit config for Colab
qw_config = '''version: 0.8
node_id: colab-indexer-1
cluster_id: quickwit-cluster
data_dir: /tmp/qwdata

storage:
  s3:
    flavor: minio
    endpoint: http://127.0.0.1:9099

metastore_uri: s3://vids-quickwit/metastore
default_index_root_uri: s3://vids-quickwit/indexes
'''
with open('/tmp/quickwit_colab.yaml', 'w') as f:
    f.write(qw_config)

print('Configured and background S3 bridge started!')

### Step 4: Run Direct Ingestion
Point `quickwit tool local-ingest` to your crawled batch NDJSON file.
It compresses, builds Tantivy splits locally in Colab RAM, and uploads directly to Google Drive!

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID'] = 'fake'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'fake'
os.environ['AWS_REGION'] = 'us-east-1'

# Run direct ingestion for your batch file
!quickwit tool local-ingest --index prn --input-path /path/to/crawled_videos.ndjson --config /tmp/quickwit_colab.yaml --yes

### Step 5: Test Search on Render
Verify that the new videos are instantly searchable via your Render public endpoint!

In [ ]:
import requests

RENDER_SEARCH_URL = 'https://vids-search-service.onrender.com/api/v1/prn/search'
query = 'Eva Elfie'

resp = requests.get(RENDER_SEARCH_URL, params={'query': query})
print(resp.json())